# lltMemoryThroughput part 3 measurements (2x T4)

Learning-rate sweep on TinyStories, baseline vs QK-norm vs z-loss at 10M, baseline vs QK-norm at 27M. One worker per T4.

In [ ]:

import io, json, os, sys, time, contextlib, platform, subprocess
RUN = {"module": "lltMemoryThroughput", "started": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()), "stdout": {}, "data": {}}

class Tee(io.StringIO):
    def write(self, s):
        sys.__stdout__.write(s)
        return super().write(s)

def run_step(key, code):
    buf = Tee()
    t = time.time()
    with contextlib.redirect_stdout(buf):
        exec(code, globals())
    RUN["stdout"][key] = buf.getvalue()
    RUN.setdefault("stepSec", {})[key] = round(time.time() - t, 1)
    print(buf.getvalue())

def save(name):
    import torch
    RUN["env"] = {"python": platform.python_version(), "torch": torch.__version__, "cuda": torch.version.cuda,
                  "gpus": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]}
    RUN["finished"] = time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())
    with open(f"/kaggle/working/{name}.json", "w") as f:
        json.dump(RUN, f, indent=1, default=str)
    print("saved", name)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'datasets', 'tokenizers'], check=False)


In [ ]:
with open('tiny.py', 'w') as f:
    f.write('# tiny.py - the lltTinyStories GPT (RMSNorm, RoPE, SwiGLU, tied embeddings) with two fixes and two probes.\nimport math\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\n\nclass RMSNorm(nn.Module):\n    def __init__(self, d, eps=1e-5):\n        super().__init__()\n        self.w, self.eps = nn.Parameter(torch.ones(d)), eps\n\n    def forward(self, x):\n        return self.w * x * torch.rsqrt(x.float().pow(2).mean(-1, keepdim=True) + self.eps).to(x.dtype)\n\n\ndef rope_tables(T, hd, device):\n    inv = 1.0 / (10000 ** (torch.arange(0, hd, 2, device=device).float() / hd))\n    f = torch.outer(torch.arange(T, device=device).float(), inv)\n    return f.cos(), f.sin()\n\n\ndef apply_rope(x, cos, sin):\n    x1, x2 = x[..., 0::2], x[..., 1::2]\n    return torch.stack([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1).flatten(-2).type_as(x)\n\n\nclass Block(nn.Module):\n    def __init__(self, d, h, qk_norm):\n        super().__init__()\n        self.h = h\n        self.n1, self.n2 = RMSNorm(d), RMSNorm(d)\n        self.qkv, self.o = nn.Linear(d, 3 * d, bias=False), nn.Linear(d, d, bias=False)\n        hid = int(8 * d / 3 / 64 + 0.999) * 64\n        self.w1, self.w3, self.w2 = nn.Linear(d, hid, bias=False), nn.Linear(d, hid, bias=False), nn.Linear(hid, d, bias=False)\n        self.qn = RMSNorm(d // h) if qk_norm else None      # QK-norm, one RMSNorm over each head\'s queries\n        self.kn = RMSNorm(d // h) if qk_norm else None      # and one over its keys\n        self.max_logit = None\n\n    def forward(self, x, cos, sin, probe=False):\n        B, T, C = x.shape\n        q, k, v = self.qkv(self.n1(x)).split(C, dim=2)\n        q, k, v = (t.view(B, T, self.h, C // self.h).transpose(1, 2) for t in (q, k, v))\n        if self.qn is not None:\n            q, k = self.qn(q), self.kn(k)\n        q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)\n        if probe:   # largest attention logit before softmax, computed in fp32\n            s = q.float() @ k.float().transpose(-2, -1) / math.sqrt(q.shape[-1])\n            causal = torch.ones(T, T, dtype=torch.bool, device=x.device).triu(1)\n            self.max_logit = s.masked_fill(causal, float("-inf")).max().item()\n        a = F.scaled_dot_product_attention(q, k, v, is_causal=True)\n        x = x + self.o(a.transpose(1, 2).reshape(B, T, C))\n        h = self.n2(x)\n        return x + self.w2(F.silu(self.w1(h)) * self.w3(h))\n\n\nclass GPT(nn.Module):\n    def __init__(self, V=4096, d=320, L=8, H=8, qk_norm=False):\n        super().__init__()\n        self.emb = nn.Embedding(V, d)\n        self.blocks = nn.ModuleList([Block(d, H, qk_norm) for _ in range(L)])\n        self.norm, self.head, self.H = RMSNorm(d), nn.Linear(d, V, bias=False), H\n        self.head.weight = self.emb.weight\n        for n, p in self.named_parameters():\n            if p.dim() == 2:\n                nn.init.normal_(p, 0.0, 0.02 / math.sqrt(2 * L) if n.endswith(("o.weight", "w2.weight")) else 0.02)\n\n    def forward(self, idx, targets, z_loss=0.0, probe=False):\n        cos, sin = rope_tables(idx.shape[1], self.emb.embedding_dim // self.H, idx.device)\n        x = self.emb(idx)\n        for b in self.blocks:\n            x = b(x, cos, sin, probe)\n        logits = self.head(self.norm(x)).float()\n        loss = F.cross_entropy(logits.flatten(0, 1), targets.flatten())\n        if z_loss or probe:\n            log_z = torch.logsumexp(logits, dim=-1)          # log of the softmax normalizer, per token\n            if z_loss:\n                loss = loss + z_loss * log_z.pow(2).mean()  # z-loss pulls log Z toward 0\n            if probe:\n                self.stats = {"maxAttnLogit": max(b.max_logit for b in self.blocks),\n                              "meanLogZ": log_z.mean().item(), "maxAbsLogit": logits.abs().max().item()}\n        return loss\n')
print('wrote', 'tiny.py')


In [ ]:
with open('sweep.py', 'w') as f:
    f.write('# sweep.py - one short TinyStories run per learning rate, with and without the fixes.\n# Usage: python sweep.py <size> <variant> <lr> [<lr> ...]   size: 10M or 27M, variant: base, qk or z\nimport json, math, sys, time\nimport numpy as np\nimport torch\nfrom tiny import GPT\n\nSIZES = {"10M": dict(d=320, L=8, H=8), "27M": dict(d=512, L=8, H=8)}\nSTEPS, WARMUP, B, T = 600, 100, 32, 256\ntrain_ids = np.load("tok_train.npy", mmap_mode="r")\nval_ids = np.load("tok_val.npy", mmap_mode="r")\n\ndef batch(arr, rng):\n    ix = rng.integers(0, arr.size - T - 1, B)\n    x = np.stack([arr[i:i + T] for i in ix]).astype(np.int64)\n    y = np.stack([arr[i + 1:i + T + 1] for i in ix]).astype(np.int64)\n    return torch.from_numpy(x).cuda(), torch.from_numpy(y).cuda()\n\n@torch.no_grad()\ndef evaluate(model):\n    model.eval()\n    rng, losses = np.random.default_rng(0), []           # the same 20 validation batches every time\n    for i in range(20):\n        x, y = batch(val_ids, rng)\n        with torch.autocast("cuda", dtype=torch.float16):\n            losses.append(model(x, y, probe=i == 0).item())\n    model.train()\n    return float(np.mean(losses)), model.stats\n\ndef run(size, variant, lr, seed=0):\n    torch.manual_seed(seed)\n    model = GPT(qk_norm=variant == "qk", **SIZES[size]).cuda()\n    z = 1e-4 if variant == "z" else 0.0\n    opt = torch.optim.AdamW(model.parameters(), lr=lr, betas=(0.9, 0.95), weight_decay=0.0)\n    scaler = torch.amp.GradScaler("cuda")\n    rng, log, skipped = np.random.default_rng(seed + 1), [], 0\n    for step in range(STEPS + 1):\n        if step % 150 == 0:\n            vl, stats = evaluate(model)\n            log.append({"step": step, "valLoss": round(vl, 4), **{k: round(v, 3) for k, v in stats.items()}})\n            print(size, variant, lr, log[-1], flush=True)\n        if step == STEPS:\n            break\n        frac = (step - WARMUP) / (STEPS - WARMUP)\n        lr_now = lr * (step + 1) / WARMUP if step < WARMUP else lr * (0.1 + 0.45 * (1 + math.cos(math.pi * frac)))\n        for g in opt.param_groups:\n            g["lr"] = lr_now\n        x, y = batch(train_ids, rng)\n        with torch.autocast("cuda", dtype=torch.float16):\n            loss = model(x, y, z_loss=z)\n        scaler.scale(loss).backward()\n        scale = scaler.get_scale()\n        scaler.step(opt)\n        scaler.update()\n        skipped += scaler.get_scale() < scale                # the scaler lowers its scale when it skips a step\n        opt.zero_grad(set_to_none=True)\n    params = sum(p.numel() for p in model.parameters())\n    return {"size": size, "params": params, "variant": variant, "lr": lr, "seed": seed, "skippedSteps": int(skipped),\n            "final": log[-1], "metrics": log}\n\nif __name__ == "__main__":\n    size, variant, lrs = sys.argv[1], sys.argv[2], [float(a) for a in sys.argv[3:]]\n    for lr in lrs:\n        t = time.time()\n        r = run(size, variant, lr)\n        r["wallClockSec"] = round(time.time() - t, 1)\n        with open(f"p3_{size}_{variant}_{lr:g}.json", "w") as f:\n            json.dump(r, f)\n')
print('wrote', 'sweep.py')


In [ ]:
with open('prep.py', 'w') as f:
    f.write('# prep.py - a 4,096-token byte-level BPE on TinyStories, and the token files sweep.py reads.\nimport time\nimport numpy as np\nfrom datasets import load_dataset\nfrom tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders\n\nds = load_dataset("roneneldan/TinyStories")\ntrain_text, val_text = ds["train"]["text"][:200_000], ds["validation"]["text"]\ntok = Tokenizer(models.BPE())\ntok.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)\ntok.decoder = decoders.ByteLevel()\ntrainer = trainers.BpeTrainer(vocab_size=4096, special_tokens=["<|eot|>"], initial_alphabet=pre_tokenizers.ByteLevel.alphabet())\nt = time.time()\ntok.train_from_iterator(train_text[:100_000], trainer=trainer)       # train split only\neot = tok.token_to_id("<|eot|>")\n\ndef encode(texts):\n    ids = []\n    for e in tok.encode_batch(texts):\n        ids.extend(e.ids)\n        ids.append(eot)\n    return np.array(ids, dtype=np.uint16)\n\nnp.save("tok_train.npy", encode(train_text))\nnp.save("tok_val.npy", encode(val_text))\nprint(f"train {np.load(\'tok_train.npy\').size:,} tokens, val {np.load(\'tok_val.npy\').size:,} tokens, {time.time() - t:.0f} s")\n')
print('wrote', 'prep.py')


In [ ]:
t = time.time()
p = subprocess.run([sys.executable, 'prep.py'], capture_output=True, text=True)
print(p.stdout, p.stderr[-2000:])
RUN['stdout']['prep'] = p.stdout
RUN['stepSec'] = {'prep': round(time.time() - t, 1)}


In [ ]:

import json, glob
LRS = ['1e-3', '3e-3', '1e-2', '3e-2', '1e-1', '3e-1']
QUEUES = {'0': [('10M', 'base'), ('10M', 'qk'), ('10M', 'z'), ('27M', 'base', 'first3')], '1': [('27M', 'qk'), ('27M', 'base', 'last3')]}
def cmds(q):
    out = []
    for item in q:
        size, variant = item[0], item[1]
        lrs = LRS if len(item) == 2 else (LRS[:3] if item[2] == "first3" else LRS[3:])
        out.append(["python", "sweep.py", size, variant] + lrs)
    return out
script = {g: " ; ".join(" ".join(c) for c in cmds(q)) for g, q in QUEUES.items()}
procs = {}
t0 = time.time()
for g, s in script.items():
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=g)
    procs[g] = subprocess.Popen(s, shell=True, env=env, stdout=open(f"worker{g}.log", "w"), stderr=subprocess.STDOUT)
for g, p in procs.items():
    p.wait()
    print("worker", g, "rc", p.returncode, round(time.time() - t0), "s")
    print(open(f"worker{g}.log").read()[-3000:])
RUN["data"]["sweep"] = [json.load(open(f)) for f in sorted(glob.glob("p3_*.json"))]
RUN["stepSec"]["sweep"] = round(time.time() - t0, 1)
save("lltMemoryThroughput_p3")


In [ ]:
run_step('read_sweep', 'import glob, json, math\n\nruns = [json.load(open(f)) for f in glob.glob("p3_*.json")]\ntable = {}\nfor r in runs:\n    table.setdefault((r["size"], r["variant"]), {})[r["lr"]] = r\nfor (size, variant), by_lr in sorted(table.items()):\n    lrs = sorted(by_lr)\n    final = [by_lr[lr]["final"] for lr in lrs]\n    start = [by_lr[lr]["metrics"][0]["valLoss"] for lr in lrs]\n    # A run that diverged counts as its step-0 loss, as in Wortsman et al. 2023\n    clipped = [min(f["valLoss"], s) if math.isfinite(f["valLoss"]) else s for f, s in zip(final, start)]\n    best = min(clipped)\n    sensitivity = sum(c - best for c in clipped) / len(clipped)\n    good = [lr for lr, c in zip(lrs, clipped) if c <= best + 0.1]\n    print(f"{size} {variant:4} val loss  " + "  ".join(f"{lr:g}={c:.3f}" for lr, c in zip(lrs, clipped)))\n    print(f"         max attention logit  " + "  ".join(f"{f[\'maxAttnLogit\']:.0f}" for f in final))\n    print(f"         mean log Z  " + "  ".join(f"{f[\'meanLogZ\']:.1f}" for f in final))\n    print(f"         LR sensitivity {sensitivity:.3f}   within 0.1 nats of best from {min(good):g} to {max(good):g}")\n')
save('lltMemoryThroughput_p3')

In [ ]:
with open('gpt.py', 'w') as f:
    f.write('# gpt.py - GPT-2 small (124M) in plain PyTorch, with the two memory switches this lab measures.\nimport math\nfrom dataclasses import dataclass\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom torch.utils.checkpoint import checkpoint\n\n\n@dataclass\nclass GPTConfig:\n    vocab: int = 50257\n    ctx: int = 1024\n    layers: int = 12\n    heads: int = 12\n    d: int = 768\n\n\nclass Block(nn.Module):\n    def __init__(self, c):\n        super().__init__()\n        self.heads = c.heads\n        self.ln1, self.ln2 = nn.LayerNorm(c.d), nn.LayerNorm(c.d)\n        self.qkv, self.proj = nn.Linear(c.d, 3 * c.d), nn.Linear(c.d, c.d)\n        self.fc, self.out = nn.Linear(c.d, 4 * c.d), nn.Linear(4 * c.d, c.d)\n\n    def forward(self, x):\n        B, T, C = x.shape\n        q, k, v = self.qkv(self.ln1(x)).split(C, dim=2)\n        q, k, v = (t.view(B, T, self.heads, -1).transpose(1, 2) for t in (q, k, v))\n        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)\n        x = x + self.proj(y.transpose(1, 2).reshape(B, T, C))\n        return x + self.out(F.gelu(self.fc(self.ln2(x)), approximate="tanh"))\n\n\ndef chunk_loss(h, w, y):\n    return F.cross_entropy((h @ w.t()).float(), y, reduction="sum")\n\n\nclass GPT(nn.Module):\n    def __init__(self, c, ckpt=False, chunks=0):\n        super().__init__()\n        self.ckpt, self.chunks = ckpt, chunks   # activation checkpointing, chunked cross-entropy\n        self.wte, self.wpe = nn.Embedding(c.vocab, c.d), nn.Embedding(c.ctx, c.d)\n        self.blocks = nn.ModuleList([Block(c) for _ in range(c.layers)])\n        self.ln_f = nn.LayerNorm(c.d)\n        self.head = nn.Linear(c.d, c.vocab, bias=False)\n        self.head.weight = self.wte.weight          # tied, as in GPT-2\n        for n, p in self.named_parameters():\n            if p.dim() == 2:\n                small = n.endswith(("proj.weight", "out.weight"))\n                nn.init.normal_(p, 0.0, 0.02 / math.sqrt(2 * c.layers) if small else 0.02)\n            elif n.endswith("bias"):\n                nn.init.zeros_(p)\n\n    def forward(self, idx, targets):\n        x = self.wte(idx) + self.wpe(torch.arange(idx.shape[1], device=idx.device))\n        for b in self.blocks:\n            x = checkpoint(b, x, use_reentrant=False) if self.ckpt else b(x)\n        h, y = self.ln_f(x).flatten(0, 1), targets.flatten()\n        if not self.chunks:\n            return F.cross_entropy(self.head(h).float(), y)\n        # Chunked cross-entropy: each chunk\'s logits are recomputed in backward, so one chunk is alive at a time\n        parts = [checkpoint(chunk_loss, hc, self.head.weight, yc, use_reentrant=False)\n                 for hc, yc in zip(h.chunk(self.chunks), y.chunk(self.chunks))]\n        return sum(parts) / y.numel()\n')
print('wrote', 'gpt.py')
with open('fsdp.py', 'w') as f:
    f.write('# fsdp.py - launch with: torchrun --nproc_per_node=2 fsdp.py  (or 1, to see one T4 run out of memory)\nimport time\nimport torch\nimport torch.distributed as dist\nfrom torch.distributed.fsdp import fully_shard, MixedPrecisionPolicy\nfrom gpt import GPT, GPTConfig\n\nCFG = GPTConfig(layers=24, heads=16, d=2048)          # about 1.3B parameters\nGB = 1e9\n\ndef build():\n    torch.manual_seed(0)\n    model = GPT(CFG, ckpt=True, chunks=8)              # built full size on the CPU\n    mp = MixedPrecisionPolicy(param_dtype=torch.float16, reduce_dtype=torch.float32)\n    for block in model.blocks:\n        fully_shard(block, mp_policy=mp)               # each block is gathered, used and freed as one unit\n    fully_shard(model, mp_policy=mp)                   # embeddings, final norm and the tied head\n    opt = torch.optim.AdamW(model.parameters(), lr=3e-4)\n    return model, opt\n\ndef train(model, opt, scaler, steps, gen, B=2, S=1024):\n    for _ in range(steps):\n        x = torch.randint(0, CFG.vocab, (B, S), device="cuda", generator=gen)\n        loss = model(x, x)\n        scaler.scale(loss).backward()\n        scaler.step(opt)\n        scaler.update()\n        opt.zero_grad(set_to_none=True)\n    return loss.item()\n\nif __name__ == "__main__":\n    dist.init_process_group("nccl")\n    rank, world = dist.get_rank(), dist.get_world_size()\n    torch.cuda.set_device(rank)\n    model, opt = build()\n    N = sum(p.numel() for p in model.parameters())\n    scaler = torch.amp.GradScaler("cuda")\n    gen = torch.Generator(device="cuda").manual_seed(100 + rank)\n    try:\n        train(model, opt, scaler, 2, gen)\n        torch.cuda.synchronize()\n        torch.cuda.reset_peak_memory_stats()\n        t = time.perf_counter()\n        loss = train(model, opt, scaler, 5, gen)\n        torch.cuda.synchronize()\n        tps = 5 * 2 * 1024 * world / (time.perf_counter() - t)\n        print(f"rank {rank} of {world}  N {N:,}  peak {torch.cuda.max_memory_allocated() / GB:.2f} GB  "\n              f"16N/{world} = {16 * N / world / GB:.2f} GB  {tps:,.0f} tokens/s  loss {loss:.3f}")\n    except torch.cuda.OutOfMemoryError:\n        print(f"rank {rank} of {world}  N {N:,}  OUT OF MEMORY at {torch.cuda.memory_allocated() / GB:.2f} GB allocated, "\n              f"16N = {16 * N / GB:.2f} GB")\n    dist.destroy_process_group()\n')
print('wrote', 'fsdp.py')
with open('fsdp_ckpt_check.py', 'w') as f:
    f.write('# fsdp_ckpt_check.py - diagnostic, not on the page. Does reloading the same checkpoint twice give\n# bit-identical losses? If yes, the 5e-5 gap between the continued and reloaded runs in fsdp_ckpt.py\n# is a restore difference. If two reloads also differ, it is run-to-run GPU nondeterminism.\nimport gc, torch\nimport torch.distributed as dist\nimport torch.distributed.checkpoint as dcp\nfrom torch.distributed.checkpoint.state_dict import get_state_dict, set_state_dict\nfrom fsdp import build, train\n\ndist.init_process_group("nccl")\nrank = dist.get_rank()\ntorch.cuda.set_device(rank)\nPATH = "/tmp/ckpt_check"\nmodel, opt = build()\nscaler = torch.amp.GradScaler("cuda")\ntrain(model, opt, scaler, 3, torch.Generator(device="cuda").manual_seed(100 + rank))\nmsd, osd = get_state_dict(model, opt)\ndcp.save({"model": msd, "optim": osd, "scaler": scaler.state_dict()}, checkpoint_id=PATH)\ndist.barrier()\nresults = {"continued": train(model, opt, scaler, 2, torch.Generator(device="cuda").manual_seed(7 + rank))}\ndel model, opt, msd, osd\ngc.collect(); torch.cuda.empty_cache()\nfor rep in ("reload1", "reload2"):\n    model, opt = build()\n    msd, osd = get_state_dict(model, opt)\n    state = {"model": msd, "optim": osd, "scaler": scaler.state_dict()}\n    dcp.load(state, checkpoint_id=PATH)\n    set_state_dict(model, opt, model_state_dict=state["model"], optim_state_dict=state["optim"])\n    scaler.load_state_dict(state["scaler"])\n    results[rep] = train(model, opt, scaler, 2, torch.Generator(device="cuda").manual_seed(7 + rank))\n    del model, opt, msd, osd, state\n    gc.collect(); torch.cuda.empty_cache()\nout = [None] * dist.get_world_size()\ndist.all_gather_object(out, {k: f"{v:.6f}" for k, v in results.items()})\nif rank == 0:\n    print("losses after 2 steps per rank:", out)\ndist.destroy_process_group()\n')
print('wrote', 'fsdp_ckpt_check.py')


In [ ]:

t = time.time()
p = subprocess.run([sys.executable, '-m', 'torch.distributed.run', '--nproc_per_node=2', 'fsdp_ckpt_check.py'], capture_output=True, text=True)
RUN['stdout']['fsdp_ckpt_check'] = p.stdout + ('' if p.returncode == 0 else p.stderr[-3000:])
RUN['stepSec']['fsdp_ckpt_check'] = round(time.time() - t, 1)
print(RUN['stdout']['fsdp_ckpt_check'])
import shutil
shutil.rmtree('/tmp/ckpt_check', ignore_errors=True)
save('lltMemoryThroughput_p3')
